<a href="https://colab.research.google.com/github/sandralilianaacosta-ui/TP-IA/blob/main/Copia_de_Copia_de_Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# TRABAJO PRÁCTICO: COMPARACIÓN DECISION TREE (PROPIO) VS RANDOM FOREST (SKLEARN)
# Inteligencia Artificial - 5° Nivel ISI - 2026

# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------------------------------
# 1. IMPLEMENTACIÓN : ÁRBOL DE DECISIÓN (C4.5)
# ------------------------------------------------------------------------------

class Node:
    """Representa un nodo dentro del árbol de decisión (de decisión o foliar/hoja)."""
    def __init__(self, feature=None, threshold=None, is_continuous=False, children=None, value=None):
        self.feature = feature          # Nombre o índice del atributo evaluado
        self.threshold = threshold      # Umbral para atributos continuos
        self.is_continuous = is_continuous
        self.children = children or {}  # Diccionario {valor_categoria: Node} o {'<=': Node, '>': Node}
        self.value = value              # Etiqueta de clase si es un nodo hoja

    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifierC45:
    """Clasificador de Árbol de Decisión basado en C4.5 con Ganancia y Gain Ratio."""
    def __init__(self, criterion='gain_ratio', max_depth=10, min_samples_split=2, threshold_gain=1e-5):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.threshold_gain = threshold_gain
        self.root = None

    def _entropy(self, y):
        """Calcula la entropía de Shannon de un vector de clases."""
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        # Evitar log2(0) definiendo 0 * log2(0) = 0
        return -np.sum([p * np.log2(p) for p in probs if p > 0])

    def _split_continuous(self, X_col, y):
        """Encuentra el mejor umbral binario para un atributo continuo."""
        best_threshold = None
        best_gain = -1.0
        best_gain_ratio = -1.0

        unique_vals = np.sort(np.unique(X_col))
        if len(unique_vals) <= 1:
            return None, -1.0, -1.0

        # Umbrales candidatos evaluados en el punto medio entre adyacentes
        thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0
        ent_D = self._entropy(y)
        n = len(y)

        for th in thresholds:
            left_mask = X_col <= th
            right_mask = ~left_mask
            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue

            ent_split = (np.sum(left_mask)/n) * self._entropy(y[left_mask]) + \
                        (np.sum(right_mask)/n) * self._entropy(y[right_mask])
            gain = ent_D - ent_split

            # Split info para el Gain Ratio
            p_left = np.sum(left_mask) / n
            p_right = np.sum(right_mask) / n
            split_info = - (p_left * np.log2(p_left) + p_right * np.log2(p_right))
            gain_ratio = (gain / split_info) if split_info > 0 else 0.0

            metric = gain_ratio if self.criterion == 'gain_ratio' else gain
            if metric > (best_gain_ratio if self.criterion == 'gain_ratio' else best_gain):
                best_gain = gain
                best_gain_ratio = gain_ratio
                best_threshold = th

        return best_threshold, best_gain, best_gain_ratio

    def _evaluate_categorical(self, X_col, y):
        """Calcula la ganancia y gain ratio para un atributo discreto/categórico."""
        n = len(y)
        ent_D = self._entropy(y)
        values, counts = np.unique(X_col, return_counts=True)

        if len(values) <= 1:
            return -1.0, -1.0

        ent_split = 0.0
        split_info = 0.0
        for val, cnt in zip(values, counts):
            sub_y = y[X_col == val]
            p = cnt / n
            ent_split += p * self._entropy(sub_y)
            split_info -= p * np.log2(p)

        gain = ent_D - ent_split
        gain_ratio = (gain / split_info) if split_info > 0 else 0.0
        return gain, gain_ratio

    def _build_tree(self, df, target_col, features, depth=0):
        """Construye recursivamente el árbol de decisión utilizando la estrategia Divide y Vencerás."""
        y = df[target_col].values

        # Criterio de parada 1: Pureza total
        if len(np.unique(y)) == 1:
            return Node(value=y[0])

        # Criterio de parada 2: No quedan atributos o profundidad máxima alcanzada
        most_common_class = pd.Series(y).mode()[0]
        if len(features) == 0 or depth >= self.max_depth or len(df) < self.min_samples_split:
            return Node(value=most_common_class)

        best_feat = None
        best_metric_val = -1.0
        best_gain = 0.0
        best_th = None
        is_cont = False

        # Evaluación de cada atributo disponible
        for feat in features:
            is_feature_numeric = np.issubdtype(df[feat].dtype, np.number)
            if is_feature_numeric and df[feat].nunique() > 5:
                # Tratar como continuo
                th, gain, gr = self._split_continuous(df[feat].values, y)
                metric = gr if self.criterion == 'gain_ratio' else gain
                if metric > best_metric_val:
                    best_metric_val = metric
                    best_gain = gain
                    best_feat = feat
                    best_th = th
                    is_cont = True
            else:
                # Tratar como discreto
                gain, gr = self._evaluate_categorical(df[feat].values, y)
                metric = gr if self.criterion == 'gain_ratio' else gain
                if metric > best_metric_val:
                    best_metric_val = metric
                    best_gain = gain
                    best_feat = feat
                    best_th = None
                    is_cont = False

        # Criterio de parada 3: La reducción de impureza no supera el umbral
        if best_feat is None or best_gain < self.threshold_gain:
            return Node(value=most_common_class)

        node = Node(feature=best_feat, threshold=best_th, is_continuous=is_cont)

        # Particionar datos y llamadas recursivas
        if is_cont:
            left_df = df[df[best_feat] <= best_th]
            right_df = df[df[best_feat] > best_th]

            # En continuo el atributo no se elimina (puede reutilizarse con otro umbral)
            node.children['<='] = self._build_tree(left_df, target_col, features, depth + 1) if len(left_df) > 0 else Node(value=most_common_class)
            node.children['>'] = self._build_tree(right_df, target_col, features, depth + 1) if len(right_df) > 0 else Node(value=most_common_class)
        else:
            remaining_features = [f for f in features if f != best_feat]
            for val in df[best_feat].unique():
                sub_df = df[df[best_feat] == val]
                node.children[val] = self._build_tree(sub_df, target_col, remaining_features, depth + 1)

        return node

    def fit(self, X, y):
        """Entrena el árbol con los datos de entrenamiento."""
        df = X.copy()
        df['__target__'] = y.values
        self.root = self._build_tree(df, '__target__', list(X.columns), depth=0)
        self.default_class = pd.Series(y).mode()[0]

    def _predict_row(self, row, node):
        if node.is_leaf():
            return node.value

        val = row[node.feature]
        if node.is_continuous:
            branch = '<=' if val <= node.threshold else '>'
            if branch in node.children:
                return self._predict_row(row, node.children[branch])
        else:
            if val in node.children:
                return self._predict_row(row, node.children[val])

        # En caso de no existir rama para un valor no visto
        return self.default_class

    def predict(self, X):
        """Genera predicciones para un DataFrame X."""
        return np.array([self._predict_row(row, self.root) for _, row in X.iterrows()])


# ------------------------------------------------------------------------------
# 2. CARGA DE DATOS Y PREPROCESAMIENTO
# ------------------------------------------------------------------------------

def load_sample_dataset():
    """Genera un dataset de demostración con atributos continuos y categóricos."""
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer(as_frame=True)
    df = data.frame.copy()
    # Tomamos 5 atributos numéricos representativos
    features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
    X = df[features]
    y = df['target'].map({0: 'Maligno', 1: 'Benigno'})
    return X, y

# Carga inicial y división en 80% train y 20% test
X_raw, y_raw = load_sample_dataset()
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw)


# ------------------------------------------------------------------------------
# 3. INTERFAZ GRÁFICA INTERACTIVA (IPYWIDGETS)
# ------------------------------------------------------------------------------

txt_title = widgets.HTML("<h2>Panel de Comparación: Decision Tree vs Random Forest</h2>")

# Controles para Decision Tree
dt_depth_slider = widgets.IntSlider(value=4, min=1, max=15, step=1, description='DT Max Prof:')
dt_criterion_dropdown = widgets.Dropdown(options=['gain_ratio', 'gain'], value='gain_ratio', description='DT Criterio:')
dt_min_split_slider = widgets.IntSlider(value=2, min=2, max=20, step=1, description='DT Min Split:')

# Controles para Random Forest
rf_n_estimators_slider = widgets.IntSlider(value=50, min=10, max=200, step=10, description='RF Estimadores:')
rf_depth_slider = widgets.IntSlider(value=4, min=1, max=15, step=1, description='RF Max Prof:')

btn_run = widgets.Button(description="Ejecutar Comparación", button_style='primary', icon='play')
output_area = widgets.Output()

def run_simulation(b):
    with output_area:
        clear_output()
        print("Entrenando modelos y evaluando métricas...")

        # 1. Decision Tree desde cero
        dt_custom = DecisionTreeClassifierC45(
            criterion=dt_criterion_dropdown.value,
            max_depth=dt_depth_slider.value,
            min_samples_split=dt_min_split_slider.value
        )
        dt_custom.fit(X_train, y_train)
        y_pred_dt = dt_custom.predict(X_test)

        # 2. Random Forest de Scikit-learn
        rf_sklearn = RandomForestClassifier(
            n_estimators=rf_n_estimators_slider.value,
            max_depth=rf_depth_slider.value,
            random_state=42
        )
        rf_sklearn.fit(X_train, y_train)
        y_pred_rf = rf_sklearn.predict(X_test)

        # 3. Métricas de Rendimiento
        labels = ['Benigno', 'Maligno']

        metrics = {
            'Métrica': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)'],
            'Decision Tree (Custom)': [
                accuracy_score(y_test, y_pred_dt),
                precision_score(y_test, y_pred_dt, average='macro'),
                recall_score(y_test, y_pred_dt, average='macro'),
                f1_score(y_test, y_pred_dt, average='macro')
            ],
            'Random Forest (Sklearn)': [
                accuracy_score(y_test, y_pred_rf),
                precision_score(y_test, y_pred_rf, average='macro'),
                recall_score(y_test, y_pred_rf, average='macro'),
                f1_score(y_test, y_pred_rf, average='macro')
            ]
        }

        df_results = pd.DataFrame(metrics)
        display(df_results.style.format({
            'Decision Tree (Custom)': '{:.4f}',
            'Random Forest (Sklearn)': '{:.4f}'
        }))

        # 4. Gráficos Comparativos
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Comparación de métricas en barras
        x = np.arange(len(df_results['Métrica']))
        width = 0.35
        axes[0].bar(x - width/2, df_results['Decision Tree (Custom)'], width, label='DT (Propio)', color='teal')
        axes[0].bar(x + width/2, df_results['Random Forest (Sklearn)'], width, label='RF (Sklearn)', color='coral')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(df_results['Métrica'], rotation=15)
        axes[0].set_ylim(0, 1.1)
        axes[0].set_title('Comparativa de Rendimiento')
        axes[0].legend()
        axes[0].grid(axis='y', linestyle='--', alpha=0.7)

        # Matriz de Confusión DT
        cm_dt = confusion_matrix(y_test, y_pred_dt, labels=labels)
        axes[1].imshow(cm_dt, cmap=plt.cm.Blues)
        axes[1].set_title(f'Matriz de Confusión - DT Custom (Acc: {accuracy_score(y_test, y_pred_dt):.2f})')
        axes[1].set_xticks([0, 1])
        axes[1].set_yticks([0, 1])
        axes[1].set_xticklabels(labels)
        axes[1].set_yticklabels(labels)
        for i in range(2):
            for j in range(2):
                axes[1].text(j, i, str(cm_dt[i, j]), ha="center", va="center", color="red" if cm_dt[i, j] < 10 else "white")

        plt.tight_layout()
        plt.show()

btn_run.on_click(run_simulation)

# Agrupación visual en interfaz
box_dt = widgets.VBox([widgets.HTML("<b>Parámetros Decision Tree</b>"), dt_depth_slider, dt_criterion_dropdown, dt_min_split_slider])
box_rf = widgets.VBox([widgets.HTML("<b>Parámetros Random Forest</b>"), rf_n_estimators_slider, rf_depth_slider])
ui = widgets.VBox([txt_title, widgets.HBox([box_dt, box_rf]), btn_run, output_area])

display(ui)
